# LegalQA v9.2 — grouped-dev candidate oracle và selector calibration
Attach source v9.2, official data, bốn approved base models, completed global dense index và QLoRA `split` artifact. Notebook dùng `selector_train.json` cho kNN nên 400 reference dev không lọt vào train-memory. Agentic được gate và chấm trên candidate-specific evidence.

In [ ]:
TRIALS = 3000
WORKING = '/kaggle/working/legalqa_selector_calibration'
AUTO_RESUME = True  # Attach đúng một selector_recovery_latest.zip để tiếp tục session trước


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
EXPECTED_SOURCE_CONTRACT = {
    'schema': 'legalqa-source-contract-v1',
    'release': '0.9.2',
    'agentic_namespace': 'agentic-v3',
    'cache_schema': 'inference-v7-agentic-error-isolation',
    'selector_schema': 'selector-weights-v3-agentic-evidence',
    'offline_wheel_count': 12,
}
REQUIRED_WHEELS = ('transformers-', 'tokenizers-', 'huggingface_hub-', 'hf_xet-', 'safetensors-', 'accelerate-', 'peft-', 'bitsandbytes-', 'sentence_transformers-', 'sentencepiece-', 'regex-', 'click-')
def source_contract_errors(path):
    errors = []
    marker = path / 'SOURCE_CONTRACT.json'
    try:
        contract = json.loads(marker.read_text(encoding='utf-8'))
    except Exception as exc:
        return [f'thiếu/hỏng SOURCE_CONTRACT.json: {exc}']
    for key, expected in EXPECTED_SOURCE_CONTRACT.items():
        if contract.get(key) != expected:
            errors.append(f'{key}={contract.get(key)!r}, cần {expected!r}')
    required_files = (path / 'kaggle' / 'run_selector_calibration.py', path / 'kaggle' / 'run_inference_only.py', path / 'src' / 'legalqa' / 'selector_schema.py', path / 'RELEASE_V9_2.md')
    errors.extend(f'thiếu {file.relative_to(path)}' for file in required_files if not file.is_file())
    wheels = list((path / 'wheelhouse').glob('*.whl'))
    names = [wheel.name for wheel in wheels]
    missing_wheels = [prefix for prefix in REQUIRED_WHEELS if not any(name.startswith(prefix) for name in names)]
    if len(wheels) != 12 or missing_wheels or any(wheel.stat().st_size < 10_000 for wheel in wheels):
        errors.append(f'wheelhouse không hợp lệ: count={len(wheels)}, missing={missing_wheels}')
    return errors
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml')})
source_reports = {str(path): source_contract_errors(path) for path in source_projects}
qualified_sources = [path for path in source_projects if not source_reports[str(path)]]
if len(qualified_sources) != 1:
    raise RuntimeError(f'Cần đúng một source v9.2 reviewed; source cũ nhúng trong QLoRA sẽ bị loại. qualified={qualified_sources}; reports={source_reports}')
selected_source = qualified_sources[0]
print('Source v9.2 reviewed:', selected_source)
print('Rejected source copies:', {path: reasons for path, reasons in source_reports.items() if reasons})
project = Path('/kaggle/working/legalqa_source')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(selected_source, project)
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], check=True)


In [ ]:
recovery_candidates = sorted(input_root.rglob('selector_recovery_latest.zip')) if AUTO_RESUME else []
if len(recovery_candidates) > 1:
    raise RuntimeError(f'Có nhiều selector recovery ZIP: {recovery_candidates}')
command = [
    sys.executable, str(project / 'kaggle' / 'run_selector_calibration.py'),
    '--input-root', '/kaggle/input',
    '--working', WORKING,
    '--trials', str(TRIALS),
]
if recovery_candidates:
    command.extend(['--resume-bundle', str(recovery_candidates[0])])
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json
root = Path(WORKING)
summary = json.loads((root / 'calibration_summary.json').read_text(encoding='utf-8'))
assert 300 <= summary['grouped_dev_count'] <= 500
assert (root / 'tuned_selection.yaml').is_file()
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Lưu tuned_selection.yaml thành Kaggle Dataset và attach vào notebook public v9.')
